PRISM-G provides functions to generate the safe and leaky reference anchors to estimate the components and privacy risk score.

### Safe anchor

The safe anchor is generated by independently sampling the genotype of every variant from a binomial distribution parameterized by the allele frequencies estimated from the real training cohort.

Specifically, for each SNP with allele frequency \(p\), synthetic genotypes are sampled as

$$
G_i \sim \mathrm{Binomial}(2, p),
$$

where $G_i \in \{0,1,2\}$ represents the genotype dosage.

Because each SNP is sampled independently, the safe anchor

- preserves allele frequencies,
- removes linkage disequilibrium,
- removes haplotype structure,
- removes individual-level relationships.

Consequently, it provides a conservative lower bound for the expected privacy risk.

The safe anchor can be generated using the `generate_safe_dataset()` function from the `utils` module of PRISM-G. It produces the same outputs as the `load_vcf())` function from the `vcf_reader` module but users can save the generated genotype matrix in VCF format using the `write_vcf()` function.

In [1]:
from prismg.io import vcf_reader as vcfio
from prismg.utils.safe_binomial import generate_safe_dataset
from prismg.io.vcf_reader import write_vcf 

# SNP Positions
LEGEND   = "../example_data/10K_SNP.legend"
legend_keys = vcfio.load_snp_legend_pos_keys(LEGEND)

safe_samples, safe_meta, safe_G = generate_safe_dataset("../example_data/1000G_10K_SNP_chr15.vcf.gz", n_safe = 2500, keep_pos = legend_keys, sample_prefix = "SAFE")
write_vcf(out_path = "../safe.vcf.gz", samples = safe_samples, meta = safe_meta, G = safe_G)

## Leaky Anchor

The leaky anchor intentionally reproduces genetic patterns from the training cohort to represent a high-risk privacy scenario.

Instead of generating completely novel genomes, the leaky anchor creates synthetic samples by copying segments from real individuals while introducing controlled perturbations. Consequently, the resulting dataset preserves much of the correlation structure and individual similarity present in the original cohort.

The leaky anchor is generated using the `generate_leaky_dataset()` function. It includes several parameters to control the degree of similarity between the generated and real datasets. For more information, please refer to the API documentation.

In [2]:
from prismg.utils.leaky_copycat import generate_leaky_dataset

leaky_samples, leaky_meta, leaky_G = generate_leaky_dataset("../example_data/1000G_10K_SNP_chr15.vcf.gz", n_samples= 2500, keep_pos = legend_keys, sample_prefix= "LEAKY")
write_vcf(out_path = "../leaky.vcf.gz", samples = leaky_samples, meta = leaky_meta, G = leaky_G)

**Note**: The leaky anchor is **not intended as a realistic synthetic data generator**. Its purpose is to provide an upper calibration point for the PRISM-G privacy score by representing an intentionally privacy-compromised dataset.